# Gns. Arbejdstid


In [10]:
from __future__ import annotations

import requests

url = "https://www.krl.dk/sirka/sirkaApi/tableApi"

# KRL payload (outputFormat='json').
# Returnerer rækker med koder: overenskomst/stilling/klassificering + nøgletal pr. måned.
# NOTE: Kun månedslønnede (afl=['0']). Kun overenskomstansatte + tjenestemænd (trb=['1','2']).
# Område-filteret (omr) sættes bredt (['1','8']).

# 'trs' afgrænser til de samme koder som Figur 3 (så vi ikke henter hele populationen).
# Selve gruppering/aggregation sker stadig via personalegrupper.py (overenskomst/stilling/klassificering).

payload = {
  "apiKey": "b6caa39c5a6dc520a83ed1624a7bdfdd1935d9fb2bee1d5e2c14dc08ee4ea57ceacca8e9550f5b25ad07b745a612d4b6b5b07bd781baff0908b5891c0beb65c1",
  "table": "Personale-måned",
  "time": [
    {"y1": "2025", "m1": "02"},
    {"y1": "2024", "m1": "02"},
    {"y1": "2023", "m1": "02"},
    {"y1": "2022", "m1": "02"},
    {"y1": "2021", "m1": "02"}
  ],
  "control": ["overenskomst", "stilling", "klassificering"],
  "data": [
    "gnsmnd", "hoveder",
  ],
  "selection": [
    {
      "name": "Udvalgte population",
      "filters": {
        "trs": [
          # Akademikere
          "272",
          # Læger (066)
          "066", "06609", "06608", "06610", "06618", "06611", "06620", "06617", "06605", "06606",
          # Læger (113)
          "113", "11304", "11310", "11301", "11308",
          # Ledende sygeplejersker
          "301", "30101",
          # Sygeplejersker
          "300", "30001", "30023", "30024", "3002302", "3002402",
          # Jordemødre
          "296", "29608", "29601", "29610", "2960103", "2961003",
          # Fysioterapeuter
          "29605", "2960102", "2961002",
          # Ergoterapeuter
          "29602", "2960101", "2961001",
          # Social- og sundhedsassistenter
          "283", "28305",
          # Sundhedsadministrativt personale
          "055", "05518", "05513", "05514", "05519",
          # Socialpædagoger
          "078", "07801",
          # Omsorgs- og pædagogmedhjælpere m.fl.
          "293",
          # Sygehusportører
          "072",
          # Rengørings- og husassistenter
          "284", "28401", "289", "28901",
          # Erhvervsuddannede serviceassistenter
          "285"
        ],
        "trb": ["1", "2"],
        "afl": ["0"],
        "omr": ["1", "8"]
      }
    }
  ],
  "options": {
    "totals": True,
    "outputFormat": "json",
    "actions": [],
    "tableName": "Beskæftigelsesgrad",
    "subLimit": 5,
    "modelName": "SIRKA",
    "timeIncreasing": True,
    "tagValues": [
      {"name": "bi1", "dsql": 31},
      {"name": "bi2", "dsql": 27},
      {"name": "bi3", "dsql": 19},
      {"name": "bi1_1", "dsql": 32},
      {"name": "bi2_1", "dsql": 28},
      {"name": "bi3_1", "dsql": 20}
    ]
  },
  "dimension": {
    "viewportHeight": 812,
    "viewportWidth": 1440,
    "xsMaxWidth": 768,
    "smMaxWidth": 992,
    "mdMaxWidth": 1200,
    "CONSTANTS": {"XS": 0, "SM": 1, "MD": 2, "LG": 3, "MAIL": 4}
  }
}

headers = {
    "Accept": "application/json,*/*;q=0.8",
    "User-Agent": "Mozilla/5.0 (Macintosh; Intel Mac OS X) Python requests",
}

response = requests.post(url, json=payload, headers=headers, timeout=180)
response.raise_for_status()

rows = response.json()
if isinstance(rows, dict) and "error" in rows:
    raise RuntimeError(rows["error"])
# Nogle svar returnerer {data: [...]} i stedet for en rå liste
if isinstance(rows, dict) and isinstance(rows.get("data"), list):
    rows = rows["data"]
if not isinstance(rows, list):
    raise RuntimeError(f"Forventede en liste med rækker, men fik: {type(rows)}")



In [11]:
import numpy as np
import pandas as pd

import sys
from pathlib import Path
import importlib

# Sørg for at mappen med 'personalegrupper.py' er på Python path
this_dir = Path.cwd()
if str(this_dir) not in sys.path:
    sys.path.insert(0, str(this_dir))

import personalegrupper as pg
pg = importlib.reload(pg)

if "rows" not in globals() or not isinstance(rows, list):
    raise RuntimeError("Variablen 'rows' mangler. Kør API-cellen først.")

years = [2021, 2022, 2023, 2024, 2025]
change_col = "Ændring fra 2021 til 2025"

display_name_overrides = {
    "Overlæger": "Overlæger*",
}

out_rows: list[dict[str, object]] = []
for group_name in pg.ORDER:
    matches = pg.COMBO_GROUPS.get(group_name)
    if matches is None:
        matches = [pg.GROUP_SPECS[group_name]]

    row_out: dict[str, object] = {"Gruppe": display_name_overrides.get(group_name, group_name)}
    vals: dict[int, float] = {}
    for y in years:
        ym = f"{y}02"
        # 'gnsmnd' er et gennemsnit i API'et; ved flere delmængder bruger vi vægtet gennemsnit (vægt=hoveder)
        v = pg.weighted_avg_matches_for_ym(rows, ym, matches, value_key="gnsmnd", weight_key="hoveder")
        vals[y] = float(v) if v is not None else np.nan
        row_out[y] = vals[y]

    if pd.notna(vals[2021]) and pd.notna(vals[2025]):
        row_out[change_col] = vals[2025] - vals[2021]
    else:
        row_out[change_col] = np.nan

    out_rows.append(row_out)

df = pd.DataFrame(out_rows).set_index("Gruppe")

def fmt_1(x) -> str:
    if pd.isna(x):
        return ""
    return f"{x:.1f}".replace(".", ",")

df_formatted = df.copy()
for y in years:
    df_formatted[y] = df[y].map(fmt_1)
df_formatted[change_col] = df[change_col].map(fmt_1)

df_formatted

,2021,2022,2023,2024,2025,Ændring fra 2021 til 2025
Gruppe,,,,,,
Akademikere,"35,3","35,1","35,1","35,0","35,0","-0,3"
Lægelige chefer,"36,1","35,8","36,3","36,5","36,3","0,2"
Overlæger*,"35,0","34,8","34,6","34,3","34,1","-0,8"
Speciallæger,"35,5","35,3","35,3","35,1","35,1","-0,5"
Uddannelseslæger,"36,2","36,0","36,1","36,1","36,1","-0,1"
Ledende sygeplejersker,"36,8","36,8","36,8","36,8","36,8","0,0"
Sygeplejersker,"33,9","33,8","33,9","33,9","34,0","0,1"
Jordemødre,"33,6","33,5","33,7","33,6","33,9","0,3"
Fysioterapeuter,"34,1","34,0","34,2","34,1","34,1","0,0"


In [13]:
# --- Preview output (compact text) ---
preview = df_formatted.copy()
preview.insert(0, 'Gruppe', preview.index)

print('COLUMNS:', list(preview.columns))
print('N_ROWS:', len(preview))
print('\nFIRST 15 ROWS:')
print(preview.head(15).to_string(index=False))
print('\nLAST 10 ROWS:')
print(preview.tail(10).to_string(index=False))


COLUMNS: ['Gruppe', 2021, 2022, 2023, 2024, 2025, 'Ændring fra 2021 til 2025']
N_ROWS: 17

FIRST 15 ROWS:
                              Gruppe 2021 2022 2023 2024 2025 Ændring fra 2021 til 2025
                         Akademikere 35,3 35,1 35,1 35,0 35,0                      -0,3
                     Lægelige chefer 36,1 35,8 36,3 36,5 36,3                       0,2
                          Overlæger* 35,0 34,8 34,6 34,3 34,1                      -0,8
                        Speciallæger 35,5 35,3 35,3 35,1 35,1                      -0,5
                    Uddannelseslæger 36,2 36,0 36,1 36,1 36,1                      -0,1
              Ledende sygeplejersker 36,8 36,8 36,8 36,8 36,8                       0,0
                      Sygeplejersker 33,9 33,8 33,9 33,9 34,0                       0,1
                          Jordemødre 33,6 33,5 33,7 33,6 33,9                       0,3
                     Fysioterapeuter 34,1 34,0 34,2 34,1 34,1                       0,0
              

In [15]:
# --- Diagnostics: confirm filters used in API payload ---
from collections.abc import Mapping, Sequence

def _find_key_paths(obj, targets, path='$'):
    paths = []
    if isinstance(obj, Mapping):
        for k, v in obj.items():
            new_path = f"{path}.{k}"
            if k in targets:
                paths.append(new_path)
            paths.extend(_find_key_paths(v, targets, new_path))
    elif isinstance(obj, Sequence) and not isinstance(obj, (str, bytes, bytearray)):
        for i, v in enumerate(obj):
            new_path = f"{path}[{i}]"
            paths.extend(_find_key_paths(v, targets, new_path))
    return paths

print('payload keys:', list(payload.keys()))
targets = {'afl', 'trb', 'omr', 'trs'}
paths = _find_key_paths(payload, targets)
print('found key paths:')
for p in paths:
    print(' -', p)

# Try common locations
filters = None
if isinstance(payload.get('filter'), dict):
    filters = payload['filter']
elif isinstance(payload.get('query'), dict) and isinstance(payload['query'].get('filter'), dict):
    filters = payload['query']['filter']
elif isinstance(payload.get('request'), dict) and isinstance(payload['request'].get('filter'), dict):
    filters = payload['request']['filter']

if filters is None:
    print('Could not auto-locate a single filter dict; see key paths above.')
else:
    print('\nExtracted filters:')
    print('afl:', filters.get('afl'))
    print('trb:', filters.get('trb'))
    print('omr:', filters.get('omr'))
    trs = filters.get('trs')
    print('trs count:', (len(trs) if isinstance(trs, list) else None))
    print('trs sample:', (trs[:10] if isinstance(trs, list) else None))


payload keys: ['apiKey', 'table', 'time', 'control', 'data', 'selection', 'options', 'dimension']
found key paths:
 - $.selection[0].filters.trs
 - $.selection[0].filters.trb
 - $.selection[0].filters.afl
 - $.selection[0].filters.omr
Could not auto-locate a single filter dict; see key paths above.
